# Analyse des délais du pipeline de planification

Ce notebook analyse les **performances et délais du pipeline de planification** de la simulation GAMA.
Il charge les fichiers de logs générés par l'expérience courante () et produit :

| Section | Description |
|---------|-------------|
| **Résumé des délais** | Statistiques (mean/std/min/max) pour chaque étape du pipeline : parsing, flagging, OTP transit, OSMnx, LLM, extraction, enqueue, WebSocket |
| **1. Débit par minute réelle** | Nombre de trajets calculés par minute de temps réel + retard de planification moyen |
| **2. Débit par heure simulée** | Nombre de trajets par heure simulée + ratio temps réel / temps simulé |
| **3. États des agents** | Évolution inactif / prêt / actif + tâches pipeline en cours au cours de la simulation |
| **4. Ponctualité des arrivées** | Répartition à l'heure / quart d'heure toulousain / en retard (source : ) |
| **4b. Agents en retard** | Historique détaillé des trajets des agents ayant ≥ 1 arrivée en retard (≥ 15 min) ou en timeout |

**Sources de données :**
-  — horodatages bruts de chaque étape du pipeline (T0 → T_fin)
-  — trajets calculés avec mode, motif et retard de planification
-  — retards réels mesurés par GAMA à l'arrivée
-  — snapshots des états des agents par sync

**Outputs :** les graphiques et CSV de synthèse sont sauvegardés dans .


Import current pipeline timing CSV

In [ ]:
LOG_DIR = '../../experiments/current/'


In [ ]:
# Parameters
LOG_DIR = "../../experiments/current/"


In [ ]:
import pandas as pd
import os


long_term_memory = False

# Chargement du fichier CSV
df = pd.read_csv(LOG_DIR + 'pipeline_timing.csv')

colonnes_not_nan = df.columns[df.isna().all()==False]
df.drop(columns=['agent_id'], inplace=True)

if long_term_memory == False:
    df.drop(columns=['T_ltm_start', 'T_ltm_end'], inplace=True)


print("Na Columns: ")
print(df.columns[df.isna().all()])


print(df.isna().sum())


def _stat_path(title, ext="csv"):
    name = "".join(c if c.isalnum() or c in "-_" else "_" for c in title)
    img_dir = os.path.join(LOG_DIR, "images/pipeline")
    os.makedirs(img_dir, exist_ok=True)
    return os.path.join(img_dir, f"{name}.{ext}")


Extract Date columns in df_delay

In [ ]:
df_delay = df.drop(columns=['P4_4_ms', 'P5_1_ms', 'P5_3_ms', 'P5_4_ms',
       'P5_5_ms', 'P5_llm_provider', 'P5_llm_retries', 'P5_tokens_in',
       'P5_tokens_out'])

df_delay

Compute delays

In [ ]:
df_delay["parse"]        = df_delay["T_parse"]       - df_delay["T0"]
df_delay["flag"]         = df_delay["T_flag"]        - df_delay["T_parse"]
df_delay["gap_otp"]      = df_delay["T_otp_start"]   - df_delay["T_flag"]
df_delay["transit_sem"]  = df_delay["T_transit_sem"] - df_delay["T_otp_start"]  # 3A queue
df_delay["transit_req"]  = df_delay["T_transit_end"] - df_delay["T_transit_sem"] # 3A HTTP
df_delay["transit"]      = df_delay["T_transit_end"] - df_delay["T_otp_start"]   # 3A total
df_delay["osmnx_sem"]    = df_delay["T_osmnx_sem"]   - df_delay["T_otp_start"]  # 3B queue (HTTP mode)
df_delay["osmnx_req"]    = df_delay["T_osmnx_end"]   - df_delay["T_osmnx_sem"]  # 3B processing (HTTP mode)
df_delay["osmnx"]        = df_delay["T_osmnx_end"]   - df_delay["T_otp_start"]  # 3B total
df_delay["otp"]          = df_delay["T_otp_end"]     - df_delay["T_otp_start"]  # 3 total = max(3A, 3B)
df_delay["gap_llm"]      = df_delay["T_llm_start"]   - df_delay["T_otp_end"]    # 3→5 transition
df_delay["llm_post"]     = df_delay["T_llm_sent"]    - df_delay["T_llm_start"]
df_delay["llm_wait"]     = df_delay["T_llm_result"]  - df_delay["T_llm_sent"]
df_delay["extract"]      = df_delay["T_extract_end"] - df_delay["T_llm_result"]
df_delay["enqueue"]      = df_delay["T_enqueue"]     - df_delay["T_extract_end"]
df_delay["ws"]           = df_delay["T_fin"]         - df_delay["T_enqueue"]
df_delay["total"]        = df_delay["T_fin"]         - df_delay["T0"]

if long_term_memory:
    df_delay["ltm"] = df_delay["T_ltm_end"] - df_delay["T_ltm_start"]

timestamp_cols = ["T_parse", "T_flag", "T_otp_start",
                  "T_transit_sem", "T_transit_end", "T_osmnx_sem", "T_osmnx_end", "T_otp_end",
                  "T_llm_start", "T_llm_sent", "T_llm_result", "T_extract_end", "T_enqueue"]
if long_term_memory:
    timestamp_cols += ["T_ltm_start", "T_ltm_end"]
df_delay = df_delay.drop(columns=[c for c in timestamp_cols if c in df_delay.columns])
df_delay

In [ ]:
delay_cols_ordered = [
    "parse",        # 1
    "flag",         # 2
    "gap_otp",      # 2→3
    "otp",          # 3 total
    "transit",      # 3A total
    "transit_sem",  # 3A-queue
    "transit_req",  # 3A-calc
    "osmnx",        # 3B total
    "osmnx_sem",    # 3B-queue (HTTP mode)
    "osmnx_req",    # 3B-calc (HTTP mode)
    "gap_llm",      # 3→5
    "llm_post",     # 5
    "llm_wait",     # 6
    "extract",      # 7
    "enqueue",      # 8
    "ws",           # 9
    "total",
]
if long_term_memory:
    delay_cols_ordered.insert(delay_cols_ordered.index("llm_post"), "ltm")

delay_cols_ordered = [c for c in delay_cols_ordered if c in df_delay.columns]

descriptions = {
    "parse":       "1     — Parsing JSON + validation Pydantic de la requête GAMA",
    "flag":        "2     — Filtrage des agents éligibles (scan de toute la population)",
    "gap_otp":     "2→3   — Délai asyncio entre lancement des tâches et démarrage OTP",
    "otp":         "3     — OTP + OSMnx gather total : durée = max(3A, 3B)",
    "transit":     "3A    — OTP transit total (queue + requête HTTP GraphQL)",
    "transit_sem": "3A-q  — Attente sémaphore OTP (queue, avant accès HTTP)",
    "transit_req": "3A-c  — Requête HTTP GraphQL OTP (calcul itinéraire transit)",
    "osmnx":       "3B    — OSMnx pied/vélo/voiture total (parallèle avec 3A)",
    "osmnx_sem":   "3B-q  — Attente sémaphore OSMnx (HTTP mode uniquement)",
    "osmnx_req":   "3B-c  — Calcul OSMnx après sémaphore (HTTP ou process pool)",
    "gap_llm":     "3→5   — Sélection candidats + construction payload LLM (non instrumenté)",
    "ltm":         "4     — Requête ChromaDB mémoire long terme (si LTM activée)",
    "llm_post":    "5     — HTTP POST création de la tâche dans le gateway LLM",
    "llm_wait":    "6     — Attente résultat LLM : long-poll Pub/Sub (micro-batch + worker + inférence)",
    "extract":     "7     — Extraction chosen_index + remapping vers le plan original (post-shuffle)",
    "enqueue":     "8     — Construction PersonMove + MoveLogger I/O + enqueue",
    "ws":          "9     — Attente envoi WebSocket vers GAMA (publish_loop)",
    "total":       "TOTAL — Durée totale pipeline T0 → T_fin",
}

delay_mean_df = df_delay[delay_cols_ordered].describe().loc[["mean", "std", "min", "max"]].T
delay_mean_df["Description"] = delay_mean_df.index.map(descriptions)
delay_mean_df.to_csv(_stat_path("pipeline_delay_summary"))
print(_stat_path("pipeline_delay_summary"))
delay_mean_df.style.format("{:.4f}", subset=["mean", "std", "min", "max"])


Véfification que l'on a toutes les valeurs

In [ ]:
# otp = total gather (transit+osmnx parallèles), transit/osmnx sont des sous-composantes
# gap_llm ferme le trou entre OTP et LLM
segment_cols = ["parse", "flag", "gap_otp", "otp", "gap_llm", "llm_post", "llm_wait", "extract", "enqueue", "ws"]
if long_term_memory:
    segment_cols.insert(segment_cols.index("llm_post"), "ltm")
segment_cols = [c for c in segment_cols if c in df_delay.columns]

df_llm = df_delay[df_delay["selection_method"] == "LLM"].copy()
df_llm["sum_segments"] = df_llm[segment_cols].sum(axis=1)
df_llm["diff"] = df_llm["total"] - df_llm["sum_segments"]

print(f"Lignes vérifiées (selection_method=LLM) : {len(df_llm)}")
print(f"Diff max  : {df_llm['diff'].abs().max():.4f} s")
print(f"Diff mean : {df_llm['diff'].abs().mean():.4f} s")
print()
df_llm[["total", "sum_segments", "diff"]].describe().style.format("{:.4f}")

---
## Analyse temporelle — Débit, ratio et états des agents

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

# --- moves.csv ---
moves_path = os.path.join(LOG_DIR, "moves.csv")
df_moves = pd.read_csv(moves_path) if os.path.exists(moves_path) else pd.DataFrame()
has_calc_time    = "Heure de calcul" in df_moves.columns
has_sim_time_col = "Temps simulé" in df_moves.columns
print(f"moves.csv : {len(df_moves)} lignes | Heure de calcul={has_calc_time} | Temps simulé={has_sim_time_col}")

# --- gama_arrivals.csv (retard réel à l'arrivée, calculé par GAMA) ---
arrivals_path = os.path.join(LOG_DIR, "gama_results", "gama_arrivals.csv")
df_arrivals = pd.read_csv(arrivals_path) if os.path.exists(arrivals_path) else pd.DataFrame()
print(f"gama_arrivals.csv : {len(df_arrivals)} lignes")

# --- agent_states.csv ---
states_path = os.path.join(LOG_DIR, "gama_results", "agent_states.csv")
df_states = pd.read_csv(states_path) if os.path.exists(states_path) else pd.DataFrame()
print(f"agent_states.csv : {len(df_states)} lignes | colonnes={list(df_states.columns) if not df_states.empty else '—'}")

In [ ]:
if df_arrivals.empty or "delay_s" not in df_arrivals.columns:
    print("gama_arrivals.csv introuvable ou colonne delay_s absente — section ignorée")
else:
    df_arrivals["delay_min"] = (df_arrivals["delay_s"] / 60).clip(lower=0).astype(int)
    if "timed_out" in df_arrivals.columns:
        df_arrivals["timed_out"] = df_arrivals["timed_out"].fillna(False).astype(bool)
    display(df_arrivals)


### 1. Débit par minute (temps réel) et retard de planification

Barres : nombre de trajets calculés par minute réelle (source : `T_fin` de pipeline_timing.csv).  
Axe secondaire : retard de planification moyen par minute (source : `Heure de calcul` + `Retard planification (s)` de moves.csv si disponibles, sinon durée pipeline totale comme proxy).

In [ ]:
# --- Binning par minute réelle (T_fin depuis pipeline_timing) ---
df_rt = df_delay[["T_fin", "sim_time", "total"]].copy()
df_rt["real_dt"] = pd.to_datetime(df_rt["T_fin"], unit="s", utc=True)
df_rt["real_min"] = df_rt["real_dt"].dt.floor("min")

per_min = (
    df_rt.groupby("real_min")
    .agg(trips=("T_fin", "count"), avg_total=("total", "mean"))
    .reset_index()
    .sort_values("real_min")
)

# Retard de planification : moves.csv (Heure de calcul) si disponible, sinon total pipeline
if has_calc_time and "Retard planification (s)" in df_moves.columns:
    _m = df_moves.copy()
    _m["real_dt"] = pd.to_datetime(_m["Heure de calcul"], utc=True)
    _m["real_min"] = _m["real_dt"].dt.floor("min")
    _delay = _m.groupby("real_min")["Retard planification (s)"].mean().rename("avg_delay")
    per_min = per_min.join(_delay, on="real_min")
    delay_label_min = "Retard planification moyen (s)"
else:
    per_min["avg_delay"] = per_min["avg_total"]
    delay_label_min = "Durée pipeline moyenne (s) [proxy]"

mean_trips_min = per_min["trips"].mean()

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

bar_width = pd.Timedelta(seconds=50)
ax1.bar(per_min["real_min"], per_min["trips"], width=bar_width,
        color="steelblue", alpha=0.75, label="Trajets calculés / min")
ax1.axhline(mean_trips_min, linestyle="--", color="steelblue", linewidth=1.4, alpha=0.7,
            label=f"Moyenne ({mean_trips_min:.0f} traj/min)")
ax2.plot(per_min["real_min"], per_min["avg_delay"],
         color="tomato", linewidth=1.8, label=delay_label_min)

ax1.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax1.xaxis.set_major_locator(mdates.MinuteLocator(interval=5))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha="right")

ax1.set_xlabel("Heure réelle")
ax1.set_ylabel("Trajets / min", color="steelblue")
ax2.set_ylabel(delay_label_min, color="tomato")
ax1.tick_params(axis="y", labelcolor="steelblue")
ax2.tick_params(axis="y", labelcolor="tomato")
ax1.set_title("Débit de calcul des trajets par minute réelle")
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left")
fig.tight_layout()
plt.savefig(_stat_path("throughput_per_real_minute", "png"), dpi=150)
plt.show()

In [ ]:
import plotly.graph_objects as go

fig_pm = go.Figure()
fig_pm.add_trace(go.Bar(
    x=per_min["real_min"], y=per_min["trips"],
    name="Trajets calculés / min", marker_color="steelblue", opacity=0.75, yaxis="y1",
))
fig_pm.add_hline(y=mean_trips_min, line_dash="dash", line_color="steelblue",
                 annotation_text=f"Moyenne ({mean_trips_min:.0f} traj/min)", yref="y1")
fig_pm.add_trace(go.Scatter(
    x=per_min["real_min"], y=per_min["avg_delay"],
    name=delay_label_min, line=dict(color="tomato", width=2), yaxis="y2",
))
fig_pm.update_layout(
    title="Débit de calcul des trajets par minute réelle",
    xaxis=dict(title="Heure réelle", tickformat="%H:%M", rangeslider=dict(visible=True)),
    yaxis=dict(title=dict(text="Trajets / min", font=dict(color="steelblue"))),
    yaxis2=dict(title=dict(text=delay_label_min, font=dict(color="tomato")), overlaying="y", side="right"),
    hovermode="x unified", height=520, legend=dict(x=0, y=1),
)
fig_pm.write_html(_stat_path("throughput_per_real_minute", "html"))
fig_pm.show()


### 2. Débit par heure simulée et retard de planification

Barres : nombre de trajets calculés par heure simulée (source : `sim_time` de pipeline_timing.csv).  
Axe secondaire : retard de planification moyen par heure simulée (source : `Temps simulé` + `Retard planification (s)` de moves.csv si disponibles, sinon durée pipeline totale comme proxy).  
L'axe X est exprimé en heures simulées relatives au début de la simulation.

In [ ]:
# --- Binning par heure simulée ---
sim_start = df_delay["sim_time"].min()

df_sh = df_delay[["sim_time", "total"]].copy()
df_sh["sim_hour"] = ((df_sh["sim_time"] - sim_start) / 3600).astype(int)

per_sim_hour = (
    df_sh.groupby("sim_hour")
    .agg(trips=("total", "count"), avg_total=("total", "mean"))
    .reset_index()
)

# Ratio temps réel / temps simulé : pour chaque heure simulée,
# on mesure la fenêtre réelle [min(T0), max(T_fin)] de toutes les tâches de cette heure
_real_span = df_delay[["T0", "T_fin", "sim_time"]].copy()
_real_span["sim_hour"] = ((_real_span["sim_time"] - sim_start) / 3600).astype(int)
_real_span = _real_span.groupby("sim_hour").agg(
    T0_min=("T0", "min"),
    T_fin_max=("T_fin", "max"),
).reset_index()
_real_span["real_elapsed"] = _real_span["T_fin_max"] - _real_span["T0_min"]
_real_span["ratio_real_sim"] = _real_span["real_elapsed"] / 3600.0
per_sim_hour = per_sim_hour.merge(_real_span[["sim_hour", "ratio_real_sim"]], on="sim_hour", how="left")

# Retard réel à l'arrivée : gama_arrivals.csv (delay_s = arrive_at - expected_arrive_at)
# Priorité : gama_arrivals > Temps simulé dans moves.csv > proxy total pipeline
if not df_arrivals.empty and "delay_s" in df_arrivals.columns:
    _arr = df_arrivals.copy()
    _arr["sim_hour"] = ((_arr["expected_arrive_at"] - sim_start) / 3600).astype(int)
    _delay_arr = (
        _arr[_arr["delay_s"] > 0]
        .groupby("sim_hour")["delay_s"]
        .mean()
        .rename("avg_delay")
    )
    per_sim_hour = per_sim_hour.join(_delay_arr, on="sim_hour")
    per_sim_hour["avg_delay"] = per_sim_hour["avg_delay"].fillna(0)/60
    delay_label_sh = "Retard arrivée moyen (min) [gama_arrivals]"
elif has_sim_time_col and "Retard planification (s)" in df_moves.columns:
    _ms = df_moves.copy()
    _ms["sim_hour"] = ((_ms["Temps simulé"] - sim_start) / 3600).astype(int)
    _delay_sh = _ms.groupby("sim_hour")["Retard planification (s)"].mean().rename("avg_delay")
    per_sim_hour = per_sim_hour.join(_delay_sh, on="sim_hour")
    per_sim_hour["avg_delay"] = per_sim_hour["avg_delay"].fillna(0)/60
    delay_label_sh = "Retard planification moyen (min) [gama_arrivals]"
else:
    per_sim_hour["avg_delay"] = per_sim_hour["avg_total"]
    delay_label_sh = "Durée pipeline moyenne (s) [proxy]"

mean_trips_sh = per_sim_hour["trips"].mean()

# Labels d'axe X : heure simulée absolue (HH:MM)
sim_hour_labels = [
    pd.Timestamp(sim_start + h * 3600, unit="s").strftime("%H:%M")
    for h in per_sim_hour["sim_hour"]
]

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax3 = ax1.twinx()
ax3.spines["right"].set_position(("outward", 70))

x = np.arange(len(per_sim_hour))
ax1.bar(x, per_sim_hour["trips"], color="mediumseagreen", alpha=0.75, label="Trajets calculés / heure simulée")
ax1.axhline(mean_trips_sh, linestyle="--", color="mediumseagreen", linewidth=1.4, alpha=0.7,
            label=f"Moyenne ({mean_trips_sh:.0f} traj/h sim)")
ax2.plot(x, per_sim_hour["avg_delay"], color="darkorange",
         linewidth=1.8, marker="o", markersize=5, label=delay_label_sh)
ax3.plot(x, per_sim_hour["ratio_real_sim"], color="purple",
         linewidth=1.8, marker="s", markersize=4, linestyle=":",
         label="Ratio temps réel / temps simulé")
ax3.axhline(1.0, linestyle="-", color="purple", linewidth=0.8, alpha=0.4)

ax1.set_xticks(x)
ax1.set_xticklabels(sim_hour_labels, rotation=45, ha="right")
ax1.set_xlabel("Heure simulée")
ax1.set_ylabel("Trajets / heure simulée", color="mediumseagreen")
ax2.set_ylabel(delay_label_sh, color="darkorange")
ax3.set_ylabel("Ratio réel / simulé", color="purple")
ax1.tick_params(axis="y", labelcolor="mediumseagreen")
ax2.tick_params(axis="y", labelcolor="darkorange")
ax3.tick_params(axis="y", labelcolor="purple")
ax1.set_title("Débit de trajets par heure simulée")
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
h3, l3 = ax3.get_legend_handles_labels()
ax1.legend(h1 + h2 + h3, l1 + l2 + l3, loc="upper left")
fig.tight_layout()
plt.savefig(_stat_path("throughput_per_sim_hour", "png"), dpi=150)
plt.show()

In [ ]:
import plotly.graph_objects as go

sim_dt_labels = [pd.Timestamp(sim_start + h * 3600, unit="s") for h in per_sim_hour["sim_hour"]]
hover_sh = [pd.Timestamp(sim_start + h * 3600, unit="s").strftime("J%d %H:%M") for h in per_sim_hour["sim_hour"]]

fig_sh = go.Figure()
fig_sh.add_trace(go.Bar(
    x=sim_dt_labels, y=per_sim_hour["trips"],
    name="Trajets / heure simulée", marker_color="mediumseagreen", opacity=0.75, yaxis="y1",
    customdata=hover_sh, hovertemplate="%{customdata}<br>%{y} trajets<extra></extra>",
))
fig_sh.add_trace(go.Scatter(
    x=sim_dt_labels, y=per_sim_hour["avg_delay"],
    name=delay_label_sh, line=dict(color="darkorange", width=2), mode="lines+markers",
    yaxis="y2", customdata=hover_sh,
    hovertemplate="%{customdata}<br>retard=%{y:.1f} min<extra></extra>",
))
fig_sh.add_trace(go.Scatter(
    x=sim_dt_labels, y=per_sim_hour["ratio_real_sim"],
    name="Ratio réel / simulé", line=dict(color="purple", width=2, dash="dot"), mode="lines+markers",
    yaxis="y3", customdata=hover_sh,
    hovertemplate="%{customdata}<br>ratio=%{y:.3f}<extra></extra>",
))
fig_sh.update_layout(
    title="Débit de trajets par heure simulée",
    xaxis=dict(title="Heure simulée", tickformat="%d/%m %H:%M", rangeslider=dict(visible=True)),
    yaxis=dict(title=dict(text="Trajets / heure simulée", font=dict(color="mediumseagreen"))),
    yaxis2=dict(title=dict(text=delay_label_sh, font=dict(color="darkorange")), overlaying="y", side="right"),
    yaxis3=dict(title=dict(text="Ratio réel / simulé", font=dict(color="purple")),
                overlaying="y", side="right", anchor="free", position=0.97),
    hovermode="x unified", height=520, legend=dict(x=0, y=1), margin=dict(r=130),
)
fig_sh.write_html(_stat_path("throughput_per_sim_hour", "html"))
fig_sh.show()


In [ ]:
per_sim_hour

### 3. Évolution des états des agents (agent_states.csv)

Graphique en aires empilées : inactif / prêt / actif sur toute la durée de la simulation.

In [ ]:
if df_states.empty:
    print("agent_states.csv introuvable — section ignorée")
else:
    # --- Graphique ---
    fig, ax1 = plt.subplots(figsize=(14, 5))

    ax1.stackplot(
        df_states["sim_timestamp"],
        df_states["inactive"],
        df_states["ready"],
        df_states["active"],
        labels=["Inactif", "Prêt", "Actif"],
        colors=["#9ecae1", "#41ab5d", "#fd8d3c"],
        alpha=0.85,
    )

    tick_step = max(1, len(df_states) // 32)
    tick_pos  = df_states["sim_timestamp"].iloc[::tick_step].values
    tick_lbl  = df_states["sim_time"].iloc[::tick_step].values
    ax1.set_xticks(tick_pos)
    ax1.set_xticklabels(tick_lbl, rotation=45, ha="right")

    ax1.set_xlabel("Heure simulée")
    ax1.set_ylabel("Nombre d'agents")
    ax1.set_title("Évolution des états des agents au cours de la simulation")

    h1, l1 = ax1.get_legend_handles_labels()
    ax1.legend(h1, l1, loc="lower right")
    fig.tight_layout()
    
    # ax1.set_xlim(650000, 750000)
    plt.savefig(_stat_path("agent_states_evolution", "png"), dpi=150)
    plt.show()

    print(df_states[["sim_time", "inactive", "ready", "active", "total"]]
          .set_index("sim_time").describe().T.to_string())

In [ ]:
import plotly.graph_objects as go

if not df_states.empty:
    states_dt = pd.to_datetime(df_states["sim_timestamp"], unit="s")
    sim_origin = df_states["sim_timestamp"].iloc[0]
    hover_day = (
        "J+" + ((df_states["sim_timestamp"] - sim_origin) // 86400).astype(str)
        + " " + df_states["sim_time"]
    )

    fig_states = go.Figure()
    for col, label, color in [
        ("inactive", "Inactif", "#9ecae1"),
        ("ready",    "Prêt",    "#41ab5d"),
        ("active",   "Actif",   "#fd8d3c"),
    ]:
        fig_states.add_trace(go.Scatter(
            x=states_dt, y=df_states[col],
            name=label, stackgroup="one",
            fillcolor=color, line=dict(color=color, width=0.5),
            customdata=hover_day,
            hovertemplate="%{customdata}<br>" + label + ": %{y}<extra></extra>",
        ))

    fig_states.update_layout(
        title="Évolution des états des agents au cours de la simulation",
        xaxis=dict(
            title="Heure simulée", tickformat="%d/%m %H:%M",
            rangeslider=dict(visible=True),
            rangeselector=dict(buttons=[
                dict(count=1,  label="1j",  step="day",  stepmode="backward"),
                dict(count=7,  label="7j",  step="day",  stepmode="backward"),
                dict(count=30, label="30j", step="day",  stepmode="backward"),
                dict(step="all", label="Tout"),
            ]),
        ),
        yaxis=dict(title="Nombre d'agents"),
        hovermode="x unified",
        height=550,
    )
    fig_states.write_html(_stat_path("agent_states_evolution", "html"))
    fig_states.show()


In [ ]:
df_states

### 4. Ponctualité des arrivées (retard réel dans GAMA)

Répartition des trajets de `gama_arrivals.csv` selon le retard d'arrivée mesuré par GAMA :
- **Arrivé à l'heure** : `delay_s ≤ 0` (arrivée avant ou à l'heure prévue)  
- **Quart d'heure toulousain** : `0 < delay_s < 15 min`  
- **En retard** : `delay_s ≥ 15 min`


In [ ]:
if df_arrivals.empty or "delay_s" not in df_arrivals.columns:
    print("gama_arrivals.csv introuvable ou colonne delay_s absente — section ignorée")
else:
    has_timeout_col = "timed_out" in df_arrivals.columns
    if has_timeout_col:
        timed_out_mask = df_arrivals["timed_out"].fillna(False).astype(bool)
        n_timeout = int(timed_out_mask.sum())
        retard = df_arrivals.loc[~timed_out_mask, "delay_s"].dropna()
    else:
        n_timeout = 0
        retard = df_arrivals["delay_s"].dropna()

    total = len(df_arrivals["delay_s"].dropna())

    a_l_heure      = (retard <= 0).sum()
    quart_toulouse = ((retard > 0) & (retard < 15 * 60)).sum()
    en_retard      = (retard >= 15 * 60).sum()

    categories = ["Arrivé à l'heure (≤ 0 s)", "Quart d'heure toulousain (< 15 min)", "En retard (≥ 15 min)"]
    counts     = [a_l_heure, quart_toulouse, en_retard]
    colors_pie = ["#41ab5d", "#fdae6b", "#e6550d"]

    if n_timeout > 0:
        categories.append("Timeout (trajet abandonné)")
        counts.append(n_timeout)
        colors_pie.append("#9467bd")

    ponctualite = pd.DataFrame({
        "Catégorie":       categories,
        "Nombre":          counts,
        "Pourcentage (%)": [100 * c / total for c in counts],
    })
    print(f"Total trajets analysés : {total}")
    if has_timeout_col:
        print(f"Dont timeouts         : {n_timeout} ({100 * n_timeout / total:.1f}%)")
    print()
    display(ponctualite.style.format({"Pourcentage (%)": "{:.1f}"}))

    fig, ax = plt.subplots(figsize=(6, 6))
    wedges, texts, autotexts = ax.pie(
        ponctualite["Nombre"],
        labels=ponctualite["Catégorie"],
        autopct="%1.1f%%",
        colors=colors_pie,
        startangle=140,
    )
    ax.set_title("Ponctualité des arrivées GAMA\n(delay_s = arrive_at − expected_arrive_at)")
    fig.tight_layout()
    plt.savefig(_stat_path("ponctualite_arrivees", "png"), bbox_inches="tight", dpi=150)
    plt.show()

    ponctualite.to_csv(_stat_path("ponctualite_arrivees"), index=False)
    print(_stat_path("ponctualite_arrivees"))


### 4b. Historique des déplacements des agents en retard (≥ 15 min)

Pour chaque agent ayant au moins un trajet en retard, on affiche l'ensemble de ses trajets
(enrichis avec le mode, le motif et l'heure de départ de `moves.csv`).


In [ ]:
if df_arrivals.empty or "delay_s" not in df_arrivals.columns:
    print("gama_arrivals.csv introuvable — section ignorée")
else:
    LATE_THRESHOLD_S = 15 * 60
    SECONDS_IN_DAY = 86400
    has_timeout_col = "timed_out" in df_arrivals.columns

    def _fmt_ts(ts):
        if pd.isna(ts):
            return ""
        s = int(ts) % SECONDS_IN_DAY
        return f"{s // 3600:02d}h{(s % 3600) // 60:02d}"

    def _fmt_delay(s):
        if pd.isna(s):
            return "N/A"
        s = int(s)
        sign = "+" if s >= 0 else "-"
        s = abs(s)
        return f"{sign}{s // 60}min{s % 60:02d}s"

    # 1. Identifier les agents avec au moins un trajet en retard ou en timeout
    late_mask = df_arrivals["delay_s"] >= LATE_THRESHOLD_S
    if has_timeout_col:
        late_mask = late_mask | df_arrivals["timed_out"].fillna(False).astype(bool)
    late_person_ids = df_arrivals.loc[late_mask, "person_id"].unique()
    print(f"Agents avec ≥1 trajet en retard (≥15 min) ou en timeout : {len(late_person_ids)}")

    # 2. Tous les trajets de ces agents
    df_hist = df_arrivals[df_arrivals["person_id"].isin(late_person_ids)].copy()

    # 3. Enrichissement via moves.csv
    cols_moves = ["ID Trajet", "Mode de transport Choisi", "Motifs de déplacement",
                  "Heure de départ", "Retard planification (s)", "Méthode de sélection"]
    cols_moves = [c for c in cols_moves if c in df_moves.columns]
    if cols_moves:
        df_hist = df_hist.merge(
            df_moves[cols_moves].rename(columns={"ID Trajet": "move_id"}),
            on="move_id", how="left"
        )

    # 4. Colonnes lisibles
    df_hist["heure prévue départ"]  = df_hist["schedule_at"].apply(_fmt_ts) if "schedule_at" in df_hist.columns else "N/A"
    df_hist["heure réelle départ"]  = df_hist["started_at"].apply(_fmt_ts) if "started_at" in df_hist.columns else "N/A"
    df_hist["retard départ GAMA"]   = df_hist["departure_delay_s"].apply(_fmt_delay) if "departure_delay_s" in df_hist.columns else "N/A"
    df_hist["arrivée prévue"]       = df_hist["expected_arrive_at"].apply(_fmt_ts)
    df_hist["arrivée réelle"]       = df_hist["arrive_at"].apply(_fmt_ts)
    df_hist["retard arrivée"]       = df_hist["delay_s"].apply(_fmt_delay)
    df_hist["🔴 retard >15min"]    = df_hist["delay_s"] >= LATE_THRESHOLD_S
    if has_timeout_col:
        df_hist["⏱️ timeout"] = df_hist["timed_out"].fillna(False).astype(bool)

    df_hist = df_hist.sort_values(["person_id", "expected_arrive_at"])

    display_cols = ["person_id",
                    "heure prévue départ", "heure réelle départ", "retard départ GAMA",
                    "arrivée prévue", "arrivée réelle", "retard arrivée",
                    "🔴 retard >15min"]
    if has_timeout_col:
        display_cols.append("⏱️ timeout")
    if "Mode de transport Choisi" in df_hist.columns:
        display_cols.insert(1, "Mode de transport Choisi")
    if "Motifs de déplacement" in df_hist.columns:
        display_cols.insert(2, "Motifs de déplacement")

    def _highlight_late(row):
        if has_timeout_col and row.get("⏱️ timeout", False):
            return ["background-color: #e8d5f5"] * len(row)
        if row["🔴 retard >15min"]:
            return ["background-color: #ffe0e0"] * len(row)
        return [""] * len(row)

    styled = (
        df_hist[display_cols]
        .style
        .apply(_highlight_late, axis=1)
        .set_caption(f"Historique complet — {len(late_person_ids)} agents concernés")
    )
    display(styled)

    df_hist[display_cols].to_csv(_stat_path("late_agents_history"), index=False)
    print(_stat_path("late_agents_history"))
